# RQ1 ② 저차원 맥락 축: k 곡선 (탐색, 피험자별)
기준은 Notion 「RQ1 판정 설계 초안」과 2026-10-05 확정 사항. 코드는 `src/fmri/axes_k_curve.py`.

**Drive 준비**: `fmri_capstone/embeddings_v2/`(배치 3 반영 후 다시 만든 것), `fmri_capstone/colab_bundle/`(새 번들), 피험자별 fMRI 폴더 `fmri_capstone/fmri/roi_betas/`(subj01, 이미 있음)와 `fmri_capstone/fmri_subj02/`, `fmri_subj05/`, `fmri_subj07/`(각각 `session01~40.npy`, `voxels.npz`, `roi_groups.npz`, `trial_map.csv`를 한 폴더에). 비공개 폴더로만 둘 것(NSD 약관).

순서: 피험자 1 탐색 → (결과 확인 후) shared1000 확인 → 피험자 2·5·7 재현. 이 노트북은 탐색과 재현을 같은 설정으로 돌린다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fmri_capstone'
!rm -rf /content/repo && git clone -q https://github.com/geon-2/fmri-imagery-context /content/repo
!pip -q install scipy
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 axes_k_curve.py --selftest | tail -3

## 피험자 1 탐색 (MPNet, ctx_P1). 학습곡선과 하위공간 안정성 포함. 시간이 꽤 걸린다(수십 분~1시간 이상)

In [ ]:
SUBJ, FMRI = 'subj01', f'{BASE}/fmri/roi_betas'
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_k_curve.py \
  --fmri-dir {FMRI} --emb-dir {BASE}/embeddings_v2 --model MPNet --rep ctx_P1 \
  --out-dir {BASE}/axes --tag {SUBJ} --boot 1000 --learning-curve --stab-boot 100

## CLIP (같은 설정)

In [ ]:
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_k_curve.py \
  --fmri-dir {FMRI} --emb-dir {BASE}/embeddings_v2 --model CLIP --rep ctx_P1 \
  --out-dir {BASE}/axes --tag {SUBJ} --boot 1000

## 그림

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
col = {'RRR': '#c44e52', 'PLS': '#55a868', 'PCA': '#4c72b0', 'RAND': '#999999'}
def plot(model, subj=SUBJ, rep='ctx_P1'):
    R = json.load(open(f'{BASE}/axes/axes_{subj}_{model}_{rep}.json'))
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
    for m, c in R['curves'].items():
        k = [x['k'] for x in c]; P = np.array([x['P'] for x in c])
        ax[0].plot(k, P, 'o-', color=col[m], label=m)
        if 'ci' in c[0]: ax[0].fill_between(k, [x['ci'][0] for x in c], [x['ci'][1] for x in c], color=col[m], alpha=.15)
    ax[0].axhline(R['full']['P'], color='k', lw=1, label='full (all dims)'); ax[0].axhline(R['tau'], color='k', ls='--', lw=.8, label='full - 1 SE')
    ax[0].set_xscale('log'); ax[0].set_xlabel('k (number of directions)'); ax[0].set_ylabel('P (median voxel r, R2/R3)')
    ax[0].set_title(f"{subj} {model} {rep}: n={R['n_images']}, k* = {R['kstar']}, dK = {R['deltaK']} {R.get('deltaK_ci','')}", fontsize=9); ax[0].legend()
    if 'learning_curve' in R:
        L = R['learning_curve']; ax[1].plot([r['n'] for r in L], [r['full_P'] for r in L], 'ko-', label='full P')
        ax[1].set_xlabel('training images'); ax[1].set_ylabel('full P'); ax[1].set_title('learning curve ' + str([(r['n'], r['kstar_RRR'], r['kstar_PCA']) for r in L]) + ' (n, k*RRR, k*PCA)', fontsize=8); ax[1].legend()
    elif 'rrr_minus_pca' in R:
        d = R['rrr_minus_pca']; ax[1].errorbar([x['k'] for x in d], [x['diff'] for x in d], yerr=[[x['diff']-x['ci'][0] for x in d],[x['ci'][1]-x['diff'] for x in d]], fmt='o-')
        ax[1].axhline(0, color='k', lw=.8); ax[1].set_xscale('log'); ax[1].set_title('RRR - PCA at the same k (95% CI)')
    plt.tight_layout(); plt.savefig(f'{BASE}/axes/axes_{subj}_{model}_{rep}.png', dpi=150); plt.show()
    if 'stability' in R:
        for s in R['stability']: print(f"k={s['k']:3d} overlap(boot)={s['overlap_mean']:.3f} {s['ci']}  chance={s['chance']:.3f}  RRR-vs-PCA overlap={s['rrr_vs_pca']:.3f}")
plot('MPNet'); plot('CLIP')

## 피험자 2·5·7 재현 (같은 설정. fMRI 폴더를 피험자별로 올린 뒤 실행)

In [ ]:
for SUBJ in ['subj02', 'subj05', 'subj07']:
    FMRI = f'{BASE}/fmri_{SUBJ}'
    for MODEL in ['MPNet', 'CLIP']:
        !cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_k_curve.py \
          --fmri-dir {FMRI} --emb-dir {BASE}/embeddings_v2 --model {MODEL} --rep ctx_P1 \
          --out-dir {BASE}/axes --tag {SUBJ} --boot 1000

## shared1000 확인 (탐색 결과를 본 뒤, 같은 json의 k*를 그대로 사용)
탐색을 마친 뒤에만 돌린다. k*를 다시 고르지 않는다.

In [ ]:
# 각 조합: 같은 탐색 결과 json의 RRR k*에서 모든 방법을 비교(k*를 다시 고르지 않는다)
for SUBJ, FMRI in [('subj01', f'{BASE}/fmri/roi_betas'), ('subj02', f'{BASE}/fmri_subj02'), ('subj05', f'{BASE}/fmri_subj05'), ('subj07', f'{BASE}/fmri_subj07')]:
    for MODEL in ['MPNet', 'CLIP']:
        print('=====', SUBJ, MODEL)
        !cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_k_curve.py \
          --fmri-dir {FMRI} --emb-dir {BASE}/embeddings_v2 --model {MODEL} --rep ctx_P1 \
          --out-dir {BASE}/axes --tag {SUBJ} --boot 1000 --confirm